In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

In [5]:
df_x_clean = pd.read_csv('../data/X_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/X_test_clean.csv')

---
## Modèle XGBoost

In [ ]:
X = df_x_clean.drop(columns=['id'])
y = df_y_clean['target']

# Split Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# stratify --> pour garder les proportions de classes


# Modèle
model = CatBoostClassifier(
    loss_function='MultiClass',
    random_seed=42,
    auto_class_weights='Balanced'
)

# Entraînement du modèle
model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
score_f1 = f1_score(y_test, y_pred, average='macro')

print(f"Score F1-Macro:{score_f1}")


Learning rate set to 0.092933
0:	learn: 1.3702361	total: 259ms	remaining: 4m 18s
1:	learn: 1.3553212	total: 308ms	remaining: 2m 33s
2:	learn: 1.3402642	total: 351ms	remaining: 1m 56s
3:	learn: 1.3277896	total: 395ms	remaining: 1m 38s
4:	learn: 1.3155587	total: 452ms	remaining: 1m 29s
5:	learn: 1.3042123	total: 538ms	remaining: 1m 29s
6:	learn: 1.2973974	total: 576ms	remaining: 1m 21s
7:	learn: 1.2870676	total: 608ms	remaining: 1m 15s
8:	learn: 1.2767777	total: 642ms	remaining: 1m 10s
9:	learn: 1.2682399	total: 671ms	remaining: 1m 6s
10:	learn: 1.2614767	total: 702ms	remaining: 1m 3s
11:	learn: 1.2553207	total: 734ms	remaining: 1m
12:	learn: 1.2481985	total: 770ms	remaining: 58.5s
13:	learn: 1.2408725	total: 809ms	remaining: 57s
14:	learn: 1.2346017	total: 842ms	remaining: 55.3s
15:	learn: 1.2301798	total: 873ms	remaining: 53.7s
16:	learn: 1.2249430	total: 902ms	remaining: 52.2s
17:	learn: 1.2184290	total: 933ms	remaining: 50.9s
18:	learn: 1.2126662	total: 962ms	remaining: 49.7s
19:	lea

In [9]:
#entraînement de ce modèle sur tout le jeu de données d'entraînement

X_train = df_x_clean.drop(columns=['id'])
y_train = df_y_clean['target']
X_test = df_x_test_clean.drop(columns=['id'])

model_final1 = CatBoostClassifier(
    loss_function='MultiClass',
    random_seed=42,
    auto_class_weights='Balanced'
)

model_final1.fit(X_train, y_train)

# prédiction sur le jeu de test
predictions = model_final1.predict(X_test)
predictions_flat = predictions.flatten()

# repasser des nombres aux noms de postes
postes = {
    0: "Assistant",
    1: "Executive",
    2: "Manager",
    3: "Director"
}
predictions_labels = [postes[p] for p in predictions_flat]

# créer le fichier
predictions_final_1 = pd.DataFrame({
    'id': df_x_test_clean['id'],
    'position': predictions_labels
})
predictions_final_1.to_csv('../results/predictions_CatBoost_1.csv', index=True)


Learning rate set to 0.093997
0:	learn: 1.3680548	total: 58.9ms	remaining: 58.9s
1:	learn: 1.3511353	total: 115ms	remaining: 57.5s
2:	learn: 1.3368567	total: 166ms	remaining: 55.3s
3:	learn: 1.3261026	total: 327ms	remaining: 1m 21s
4:	learn: 1.3140257	total: 375ms	remaining: 1m 14s
5:	learn: 1.3031218	total: 422ms	remaining: 1m 9s
6:	learn: 1.2966457	total: 462ms	remaining: 1m 5s
7:	learn: 1.2864952	total: 498ms	remaining: 1m 1s
8:	learn: 1.2753383	total: 536ms	remaining: 59.1s
9:	learn: 1.2674328	total: 588ms	remaining: 58.2s
10:	learn: 1.2603975	total: 646ms	remaining: 58.1s
11:	learn: 1.2521757	total: 698ms	remaining: 57.5s
12:	learn: 1.2487472	total: 738ms	remaining: 56s
13:	learn: 1.2405641	total: 786ms	remaining: 55.4s
14:	learn: 1.2332194	total: 834ms	remaining: 54.7s
15:	learn: 1.2259572	total: 887ms	remaining: 54.6s
16:	learn: 1.2208877	total: 937ms	remaining: 54.2s
17:	learn: 1.2155454	total: 986ms	remaining: 53.8s
18:	learn: 1.2102160	total: 1.04s	remaining: 53.8s
19:	learn:

Votre score pour cette soumission est : 

In [10]:
# les 32 colonnes d'entreprise
company_cols = [f'company_feature_{i}' for i in range(32)]

#regroupe sous un même identifiant les lignes qui ont la meme entreprise
df_x_clean['company_group'] = df_x_clean[company_cols].apply(lambda row: hash(tuple(row)), axis=1)
groups_company = df_x_clean['company_group']


#on va faire une validation croisée en utilisant StratifiedGroupKFold
#le découpage respecte à la fois les groupes d'entités et l'équilibre des classes dans chaque bloc.

X = df_x_clean.drop(columns=['id', 'company_group'])
y = df_y_clean['target']

cv = StratifiedGroupKFold(n_splits=5,random_state=42, shuffle=True)
scores_f1 = []

#entraînement et d'évaluation pli par pli
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups=groups_company)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    model = CatBoostClassifier(
        loss_function='MultiClass',
        random_seed=42,
        auto_class_weights='Balanced'
    )
    
    model.fit(X_tr, y_tr)
    
    preds = model.predict(X_te)
    score = f1_score(y_te, preds, average='macro')
    scores_f1.append(score)
    
    print(f"Pli {fold + 1} - F1-Macro : {score}")

print(f"\n---> F1-Macro Moyen : {np.mean(scores_f1)}")


    

Learning rate set to 0.092933
0:	learn: 1.3696201	total: 35.7ms	remaining: 35.7s
1:	learn: 1.3538303	total: 65ms	remaining: 32.4s
2:	learn: 1.3358581	total: 95.4ms	remaining: 31.7s
3:	learn: 1.3217602	total: 133ms	remaining: 33.2s
4:	learn: 1.3077789	total: 168ms	remaining: 33.5s
5:	learn: 1.2949122	total: 200ms	remaining: 33.1s
6:	learn: 1.2878437	total: 232ms	remaining: 32.9s
7:	learn: 1.2778921	total: 269ms	remaining: 33.3s
8:	learn: 1.2646358	total: 309ms	remaining: 34s
9:	learn: 1.2548395	total: 341ms	remaining: 33.7s
10:	learn: 1.2461498	total: 372ms	remaining: 33.5s
11:	learn: 1.2368534	total: 405ms	remaining: 33.4s
12:	learn: 1.2314468	total: 436ms	remaining: 33.1s
13:	learn: 1.2214491	total: 475ms	remaining: 33.5s
14:	learn: 1.2153581	total: 516ms	remaining: 33.9s
15:	learn: 1.2101215	total: 556ms	remaining: 34.2s
16:	learn: 1.2006236	total: 596ms	remaining: 34.5s
17:	learn: 1.1927554	total: 640ms	remaining: 34.9s
18:	learn: 1.1879220	total: 681ms	remaining: 35.2s
19:	learn: 1

## Recherche hyperparamètre CatBoost

In [ ]:
# les 32 colonnes d'entreprise
company_cols = [f'company_feature_{i}' for i in range(32)]

#regroupe sous un même identifiant les lignes qui ont la meme entreprise
df_x_clean['company_group'] = df_x_clean[company_cols].apply(lambda row: hash(tuple(row)), axis=1)
groups_company = df_x_clean['company_group']

#on va faire une validation croisée en utilisant StratifiedGroupKFold
#le découpage respecte à la fois les groupes d'entités et l'équilibre des classes dans chaque bloc.

X = df_x_clean.drop(columns=['id', 'company_group'])
y = df_y_clean['target']


# hyperparamètres à tester
param_test = {
    'iterations': [200, 400, 600, 800],
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.03, 0.08, 0.15],
    'l2_leaf_reg': [1, 3, 5, 10, 20],
    'subsample': [0.6, 0.8, 1.0],
    'grow_policy': ['SymmetricTree', 'Depthwise']
}



model1 = CatBoostClassifier(
    loss_function='MultiClass',
    random_seed=42,
    auto_class_weights='Balanced'
)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# recherche aléatoire hyperparamètres
recherche = RandomizedSearchCV(
    estimator=model1,
    param_distributions=param_test,
    n_iter=20,
    scoring='f1_macro',
    cv=cv,
    random_state=42,
    n_jobs=-1
)
recherche.fit(X, y, groups=groups_company)


print(f"--- Meilleur F1-Macro Réel (GroupKFold) : {recherche.best_score_:.4f} ---")
print("\nMeilleurs hyperparamètres trouvés :")
print(recherche.best_params_)

--- Meilleur F1-Macro Réel (GroupKFold) : 0.3576 ---

Meilleurs hyperparamètres trouvés :
{'subsample': 0.85, 'reg_lambda': 5, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 0.7}


In [ ]:
#entraînement du modéle avec les meilleurs hyperparamètres sur tout le jeu de données d'entraînement

X_train = pd.read_csv('../data/X_train_clean.csv')
X_train = X_train.drop(columns=['id'])
y_train = df_y_clean['target']
X_test = df_x_test_clean.drop(columns=['id'])


model_final2 = CatBoostClassifier(
    loss_function='MultiClass',
    random_seed=42,
    auto_class_weights='Balanced',
    iterations=recherche.best_params_['iterations'],
    depth=recherche.best_params_['depth'],
    learning_rate=recherche.best_params_['learning_rate'],
    l2_leaf_reg=recherche.best_params_['l2_leaf_reg'],
    subsample=recherche.best_params_['subsample'],
    grow_policy=recherche.best_params_['grow_policy']
)

model_final2.fit(X_train, y_train)

# prédiction sur le jeu de test
predictions = model_final2.predict(X_test)

# repasser des nombres aux noms de postes
postes = {
    0: "Assistant",
    1: "Executive",
    2: "Manager",
    3: "Director"
}
predictions_labels = [postes[p] for p in predictions]

# créer le fichier
predictions_final_2 = pd.DataFrame({
    'id': df_x_test_clean['id'],
    'position': predictions_labels
})
predictions_final_2.to_csv('../results/predictions_CatBoost_2.csv', index=True)